In [ ]:
# %%
import pandas as pd 
import numpy as np
# import matplotlib.pyplot as plt 
# import seaborn as sns
import torch
from functools import partial
# from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../"))
if project_root not in sys.path:
    sys.path.append(project_root)

# Load .env from project root
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, ".env"))

# Read model config from environment
LLM_MODEL = os.environ["LLM_MODEL"]
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "nomic-ai/nomic-embed-text-v1.5")
LLM_BASE_URL = os.environ.get("LLM_BASE_URL", "http://127.0.0.1:8080/v1")
EMBED_BASE_URL = os.environ.get("EMBED_BASE_URL", "http://127.0.0.1:8081/v1")

print(f"LLM_MODEL     : {LLM_MODEL}")
print(f"EMBEDDING_MODEL: {EMBEDDING_MODEL}")
print(f"LLM_BASE_URL  : {LLM_BASE_URL}")
print(f"EMBED_BASE_URL : {EMBED_BASE_URL}")

import asyncio
from lightrag.utils import setup_logger, EmbeddingFunc
from lightrag.llm.hf import hf_embed
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import gpt_4o_mini_complete, gpt_4o_complete, openai_embed, openai_complete
from lightrag.utils import setup_logger

# %%
setup_logger("lightrag", level="INFO")

WORKING_DIR = "./rag_storage"
if not os.path.exists(WORKING_DIR):
    os.mkdir(WORKING_DIR)

In [ ]:
async def hf_model_complete(prompt: str, system_prompt=None, history_messages=[], **kwargs):
    device = model.device
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})
    
    # Use chat template to wrap the complex LightRAG instructions
    tokenized_chat = llm_tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
    inputs = {"input_ids": tokenized_chat.to(device)}

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=True,
        pad_token_id=llm_tokenizer.eos_token_id
    )

    decoded = llm_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return decoded.strip()

In [ ]:
async def llm_complete(prompt, system_prompt=None, history_messages=None, **kwargs):
    """LLM completion using the model configured in .env (LLM_MODEL)."""
    kwargs.update({
        "temperature": 0.1,
        "top_p": 0.95,
        "max_tokens": 1024,
    })

    return await openai_complete(
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        **kwargs
    )

In [ ]:
# Using vLLM — model name and URLs come from .env

rag = LightRAG(
    working_dir=WORKING_DIR,
    llm_model_func=llm_complete,
    llm_model_name=LLM_MODEL,
    llm_model_max_async=4,
    llm_model_kwargs={
        "base_url": LLM_BASE_URL,
        "api_key": "none"
    },
    chunk_token_size=1200,
    entity_extract_max_gleaning=0,
    default_embedding_timeout=120,
    
    embedding_func=EmbeddingFunc(
        embedding_dim=768, 
        max_token_size=8192,
        func=partial(
            openai_embed.func,
            base_url=EMBED_BASE_URL,
            api_key="none",
            model=EMBEDDING_MODEL
        )
    )
)

In [ ]:
await rag.initialize_storages()

In [ ]:
data_dir = os.path.join(project_root, 'biomed-rag', 'data', 'external', 'medqa')
sample_textbook = os.path.join(data_dir, 'textbooks', 'Anatomy_Gray.txt')

with open(sample_textbook, 'r') as f:
    text = f.read()

In [ ]:
await rag.ainsert(text)

## 🧪 Model Test Block

Use the cells below to test the LLM and embedding servers independently, **without** needing a full RAG pipeline.

In [ ]:
# --- Test: Direct LLM call via OpenAI-compatible API ---
from openai import AsyncOpenAI

llm_client = AsyncOpenAI(base_url=LLM_BASE_URL, api_key="none")

TEST_PROMPT = "What are the main causes of hypertension? Answer in 3 bullet points."
TEST_SYSTEM  = "You are a helpful medical assistant."

response = await llm_client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": TEST_SYSTEM},
        {"role": "user",   "content": TEST_PROMPT},
    ],
    temperature=0.1,
    max_tokens=512,
)

print(f"Model  : {response.model}")
print(f"Tokens : prompt={response.usage.prompt_tokens}, completion={response.usage.completion_tokens}")
print("\n--- Response ---")
print(response.choices[0].message.content)

In [ ]:
# --- Test: Embedding server ---
from openai import AsyncOpenAI

embed_client = AsyncOpenAI(base_url=EMBED_BASE_URL, api_key="none")

TEST_TEXT = "The heart pumps blood throughout the body."

embed_response = await embed_client.embeddings.create(
    model=EMBEDDING_MODEL,
    input=TEST_TEXT,
)

vec = embed_response.data[0].embedding
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Vector dimension: {len(vec)}")
print(f"First 8 values  : {vec[:8]}")

In [ ]:
# --- Test: RAG query (requires storages to be initialized and documents inserted) ---
QUERY = "What is the function of the aorta?"
MODE  = "hybrid"  # options: naive | local | global | hybrid

result = await rag.aquery(QUERY, param=QueryParam(mode=MODE))
print(f"Query : {QUERY}")
print(f"Mode  : {MODE}")
print("\n--- Answer ---")
print(result)